In [ ]:
import os
import json
import random
import torch
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Change this (or set the FLOWER_DATA_ROOT environment variable) to point at
# your local dataset root if you are not running this on Kaggle.
# Expected layout: DATA_ROOT/cat_to_name.json, DATA_ROOT/dataset/{train,valid,test}
DATA_ROOT = os.environ.get("FLOWER_DATA_ROOT", "/kaggle/input/dataset")

In [ ]:
# Optional: explore the Kaggle input directory layout
for dirname, _, _ in os.walk('/kaggle/input'):
    print(dirname)

## Load category-to-name mapping
Loads `cat_to_name.json`, mapping category numbers to flower names.

In [ ]:
cat_to_name_path = os.path.join(DATA_ROOT, "cat_to_name.json")

with open(cat_to_name_path, "r") as f:
    cat_to_name = json.load(f)

# Convert string keys to integers (useful for indexing)
cat_to_name = {int(k): v for k, v in cat_to_name.items()}

print(f"Loaded {len(cat_to_name)} categories.")

# Resize all images to 224x224 to match CNN input size
# Convert images to PyTorch tensors
# Normalize images using ImageNet mean and standard deviation

In [ ]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(30),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Define transformations for validation and test images (no augmentation)
valid_test_transforms = transforms.Compose([
    transforms.Resize(256),                # Resize shortest side to 256
    transforms.CenterCrop(224),            # Crop the center region
    transforms.ToTensor(),                 # Convert image to tensor
    transforms.Normalize(                  # Normalize pixel values
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load training images from directory and apply transformations
# Load validation images
# Load test images

In [ ]:
train_dir = os.path.join(DATA_ROOT, "dataset", "train")
test_dir  = os.path.join(DATA_ROOT, "dataset", "test")
valid_dir = os.path.join(DATA_ROOT, "dataset", "valid")

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load training dataset with transformations
train_data = datasets.ImageFolder(train_dir, transform=train_transforms)

# Load validation dataset
valid_data = datasets.ImageFolder(valid_dir, transform=valid_test_transforms)

In [ ]:
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp')

# Custom Dataset class for test images (unlabeled, used for the Kaggle submission)
class TestDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir            # Directory containing images
        self.transform = transform          # Transformations to apply
        self.images = sorted(               # Only keep actual image files
            f for f in os.listdir(root_dir)
            if f.lower().endswith(IMAGE_EXTENSIONS)
        )

    def __len__(self):
        # Return total number of images
        return len(self.images)

    def __getitem__(self, idx):
        # Get image filename by index
        img_name = self.images[idx]
        img_path = os.path.join(self.root_dir, img_name)

        # Open image, forcing 3-channel RGB (some images may be grayscale/RGBA)
        image = Image.open(img_path).convert("RGB")

        # Apply transformations if defined
        if self.transform:
            image = self.transform(image)

        # Return image and its filename
        return image, img_name


# Create test dataset object
test_dataset = TestDataset(
    root_dir=test_dir,
    transform=valid_test_transforms
)

In [ ]:
# Function to save preprocessed dataset to disk
def save_preprocessed_dataset(dataset, name, save_dir='preprocessed'):
    os.makedirs(save_dir, exist_ok=True)  # Create directory if it doesn't exist

    data_list = []

    # Loop through dataset and store (image, label) pairs
    for img, label in dataset:
        data_list.append((img, label))

    # Save dataset as a PyTorch file
    torch.save(data_list, f'{save_dir}/{name}.pt')

    # Print confirmation message
    print(f"✅ Saved {len(data_list)} {name} samples")

In [ ]:
print("Number of training images:", len(train_data))
print("Number of validation images:", len(valid_data))
print("Number of test images:", len(test_dataset))
print("Number of classes:", len(train_data.classes))

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ImageFolder assigns label indices by alphabetically sorting class folder
# names ("1", "10", "100", ... "2", "20", ...), which is NOT the same order
# as the numeric category id. This maps the label index back to the real
# category id used by cat_to_name.json.
idx_to_class = {v: int(k) for k, v in train_data.class_to_idx.items()}

def get_flower_name(class_index):
    return cat_to_name[idx_to_class[class_index]]

images, labels = next(iter(train_loader))
img = images[0].permute(1, 2, 0).numpy()
img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
img = np.clip(img, 0, 1)

plt.imshow(img)
plt.title(get_flower_name(labels[0].item()))
plt.axis("off")
plt.show()

# Save all datasets after preprocessing
save_preprocessed_dataset(train_data, 'train')
save_preprocessed_dataset(valid_data, 'valid')
save_preprocessed_dataset(test_dataset, 'test')

# Persist the label-index -> real category-id mapping alongside the cached
# tensors so downstream notebooks/scripts can correctly map model
# predictions back to flower names (the .pt files only store raw indices).
with open('preprocessed/idx_to_class.json', 'w') as f:
    json.dump({str(k): v for k, v in idx_to_class.items()}, f)
print("Saved preprocessed/idx_to_class.json")